In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small/checkpoints/post_cell_27_try_4.pickle

In [ ]:
%%cudf.pandas.profile
### cell 28 ###

# 1. Standardize question prefixes and answer labels across all DataFrames
orig_questions = [
    'Which of the following hosted notebook products do you use on a regular basis?',
    'Which of the following hosted notebooks have you used at work or school in the last 5 years?'
]
new_question = 'Do you use any of the following hosted notebook products?'
label_replacements = {
    'Google Colab ':            'Colab Notebooks',
    'Kaggle Notebooks (Kernels) ': 'Kaggle Notebooks',
    'Kaggle Kernels':            'Kaggle Notebooks'
}

dfs_to_clean = [
    responses_df_2017,
    responses_df_2018_cell10,
    responses_df_2019_cell10,
    responses_df_2020,
    responses_df_2021,
    responses_df_2022_cell10
]
for df in dfs_to_clean:
    cols = df.columns
    # replace any of the original question strings with the new one
    for oq in orig_questions:
        cols = cols.str.replace(oq, new_question, regex=False)
    # standardize answer labels in column names
    for old, new in label_replacements.items():
        cols = cols.str.replace(old, new, regex=False)
    df.columns = cols

# 2. Subset + rename answers + drop all-NaN rows in one pass
#    We split on all dashes and take the last piece to get just the answer label

def grab_subset_of_data_40(original_df, question):
    subset_cols = [c for c in original_df.columns if question in c]
    mapper = [c.split('-')[-1].strip() for c in subset_cols]
    return (
        original_df[subset_cols]
                   .rename(columns=dict(zip(subset_cols, mapper)))
                   .dropna(how='all', subset=mapper)
    )

# 3. Combine yearly subsets, assign year and count responses
def combine_subsets(question, include_2017=False):
    year_sources = [
        (2018, responses_df_2018_cell10),
        (2019, responses_df_2019_cell10),
        (2020, responses_df_2020),
        (2021, responses_df_2021),
        (2022, responses_df_2022_cell10)
    ]
    if include_2017:
        year_sources.insert(0, (2017, responses_df_2017))
    pieces = []
    for yr, df_src in year_sources:
        sub = grab_subset_of_data_40(df_src, question)
        sub['year'] = str(yr)
        pieces.append(sub)
    combined = pd.concat(pieces, ignore_index=True)
    counts = combined.groupby('year').count().reset_index()
    return combined, counts

# 4. Execute combine and compute percentages via vectorized operations
topic = new_question
notebooks_df_combined, notebooks_df_combined_counts = combine_subsets(topic)
# total respondents per year
totals = notebooks_df_combined['year'].value_counts().sort_index()
# compute percentages
notebooks_df_combined_percentages = (
    notebooks_df_combined_counts
      .set_index('year')
      .div(totals, axis=0)
      .mul(100)
      .reset_index()
)

# 5. Pivot for plotting
df_cell40 = (
    notebooks_df_combined_percentages
      .loc[:, ['year', 'None', 'Kaggle Notebooks', 'Colab Notebooks']]
      .melt(id_vars=['year'], value_vars=['None', 'Kaggle Notebooks', 'Colab Notebooks'])
      .rename(columns={'variable': ''})
)
# inspect
df_cell40.info()
